In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import importlib
import random
import dask.dataframe as dd
import pandas as pd
import numpy as np
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster
import statsmodels.api as sm
from sqlalchemy.orm import aliased
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

# Define trading parameters for OU model
STOP_LOSS_FACTOR = 2.25
DISCOUNT_RATE = 0.0001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
P_VALUE_THRESHOLD = 0.01  # Only trade if p_value < 0.01 (99% confidence)
CLUSTER_TYPE = "local"
N_WORKERS = 4


engine = create_engine(POSTGRES_URL)

In [ ]:
max_groups = 5
window_days = 7
window = window_days * 24 * 60
test_days = 7
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

In [ ]:
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).where(
            models.ProviderAssetGroup.is_active.is_(True)
        )
    ).all()
    provider_asset_group_ids = random.sample(provider_asset_group_ids, max_groups)
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
# Load provider asset group members (filtered)
print("Loading provider asset group members...")
members_data = pd.read_sql(
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    ).where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    ),
    engine,
)
print(f"Members data loaded: {len(members_data)} rows")

# Load market data with pandas (before cluster)
print("Loading market data...")
market_data = pd.read_sql(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    )
    .where(
        models.ProviderAssetMarket.timestamp.between(start_time, end_time),
        models.ProviderAssetMarket.from_asset_id.in_(
            members_data["from_asset_id"].astype(int).unique().tolist()
        ),
        models.ProviderAssetMarket.to_asset_id.in_(
            members_data["to_asset_id"].astype(int).unique().tolist()
        ),
    )
    .order_by(models.ProviderAssetMarket.timestamp),
    engine,
)
print(f"Market data loaded: {len(market_data)} rows")

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="6GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="coiled-cluster",
        n_workers=N_WORKERS,
        region="us-east-1",
        worker_memory="16GB",
        worker_cpu=4,
    )

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
@delayed
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    members_chunk: pd.DataFrame,
    market_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load the pairs trading frame for a chunk of provider asset groups.
    Returns only the essential columns needed for cointegration analysis.

    Args:
        provider_asset_group_ids: List of provider asset group IDs to process
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        members_chunk: Pre-filtered DataFrame of provider asset group members
        market_data: Broadcasted market data DataFrame

    Returns:
        pandas DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Step 1: Generate timeframe using pd.date_range
    time_frame = pd.DataFrame({"timestamp": pd.date_range(start, end, freq="1min")})

    # Step 2: Use passed-in members_chunk (already filtered)
    members = members_chunk

    # Step 3: Cross join
    full_frame = time_frame.merge(members, how="cross")
    full_frame = full_frame.sort_values("timestamp")

    # Step 4: Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame,
        market_data,
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Step 5: Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})
    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    members_data: pd.DataFrame,
    market_data_future,
    n_workers: int = 10,
) -> dd.DataFrame:
    """
    Get the pairs trading frame with only essential columns for cointegration analysis.

    Args:
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        provider_asset_group_ids: List of provider asset group IDs to process
        members_data: Pre-loaded DataFrame of provider asset group members
        market_data_future: Broadcasted market data future
        n_workers: Number of parallel workers

    Returns:
        Dask DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Split provider asset groups into chunks
    provider_asset_group_ids = sorted(provider_asset_group_ids)
    n_chunks = max(n_workers, len(provider_asset_group_ids))
    group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

    # Create delayed tasks with filtered member chunks
    delayed_dfs = []
    for chunk in group_chunks:
        # Filter members data for this specific chunk
        members_chunk = members_data[
            members_data["provider_asset_group_id"].isin(chunk.tolist())
        ].copy()

        delayed_dfs.append(
            load_pairs_trading_frame_chunk(
                start, end, members_chunk, market_data_future
            )
        )

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)
    pairs_trading_frame = pairs_trading_frame.reset_index()

    # Set index to provider_asset_group_id
    pairs_trading_frame = pairs_trading_frame.set_index(
        "provider_asset_group_id", sorted=True
    )

    return pairs_trading_frame

In [ ]:
pairs_trading_frame = get_pairs_trading_frame(
    start_time,
    end_time,
    provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
def rolling_cointegration(
    timestamp: pd.Series, close_1: np.ndarray, close_2: np.ndarray, window: int
) -> pd.DataFrame:
    result = stochastic.RollingCointegration(
        close_1,
        close_2,
        window=window,
    ).fit()
    n = len(timestamp)
    timestamp_values = timestamp.values if hasattr(timestamp, "values") else timestamp
    return pd.DataFrame(
        {
            "timestamp": timestamp_values,
            "close_1": close_1,
            "close_2": close_2,
            "alpha": result.alpha,
            "beta": result.beta,
            "pvalue": result.pvalue,
        },
        index=pd.RangeIndex(n),  # Explicit integer index to avoid reindexing issues
    )

In [ ]:
pairs_trading_fitted_frame = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["timestamp", "close_1", "close_2"]
].apply(
    lambda df: rolling_cointegration(
        df["timestamp"], df["close_1"].to_numpy(), df["close_2"].to_numpy(), window
    ),
    meta={
        "timestamp": pd.Series([], dtype="datetime64[ns]"),
        "close_1": pd.Series([], dtype=float),
        "close_2": pd.Series([], dtype=float),
        "alpha": pd.Series([], dtype=float),
        "beta": pd.Series([], dtype=float),
        "pvalue": pd.Series([], dtype=float),
    },
)

In [ ]:
pairs_trading_fitted_frame_computed = pairs_trading_fitted_frame.compute()
pairs_trading_fitted_frame_computed

In [ ]:
pairs_trading_fitted_frame_computed = pairs_trading_fitted_frame_computed.loc[
    pairs_trading_fitted_frame_computed["alpha"].notnull()
]

In [ ]:
pairs_trading_fitted_frame_computed = pairs_trading_fitted_frame_computed.reset_index(
    level="provider_asset_group_id"
)
pairs_trading_fitted_frame_computed

In [ ]:
def rolling_ornstein_uhlenbeck(
    timestamp: pd.Series,
    alpha: np.ndarray,
    beta: np.ndarray,
    pvalue: np.ndarray,
    window: int,
) -> pd.DataFrame:
    result = stochastic.RollingOrnsteinUhlenbeck(
        alpha, beta, pvalue, window=window
    ).fit()
    return result

In [ ]:
pairs_trading_fitted_frame_computed.to_parquet(
    "pairs_trading_fitted_frame_computed.parquet"
)

In [ ]:
pairs_trading_fitted_frame_computed = pd.read_parquet(
    "pairs_trading_fitted_frame_computed.parquet"
)
pairs_trading_fitted_frame_computed

In [ ]:
pairs_trading_fitted_frame_computed

In [ ]:
pairs_trading_fitted_frame_computed["ou_L"].to_numpy()

In [ ]:
example = pairs_trading_fitted_frame_computed.loc[
    pairs_trading_fitted_frame_computed["ou_mu"].notna()
]
example

In [ ]:
importlib.reload(stochastic)

In [ ]:
analytical = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
    mu=example["ou_mu"].to_numpy(),
    sigma=example["ou_sigma"].to_numpy(),
    theta=example["ou_theta"].to_numpy(),
    L=example["ou_theta"].to_numpy()
    - example["ou_sigma"].to_numpy() * STOP_LOSS_FACTOR,
    r=0.01,
    c=0.001,
    use_analytical=True,
)
analytical

In [ ]:
numerical = stochastic.OrnsteinUhlenbeck.F(
    (example["ou_theta"] - example["ou_sigma"] * STOP_LOSS_FACTOR).to_numpy(),
    example["ou_mu"].to_numpy(),
    example["ou_sigma"].to_numpy(),
    example["ou_theta"].to_numpy(),
    0.01,
    use_analytical=False,
)
numerical

In [ ]:
"""
Code to plot how numerical vs analytical F(x) relationship changes across different inputs.
Can be added as a notebook cell.
"""

import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add the workspace root to Python path
workspace_root = Path.cwd().parent if Path.cwd().name != "mc-notebooks" else Path.cwd()
sys.path.insert(0, str(workspace_root))

import src.utils.stochastic as stochastic

# Parameters
r = 0.0001
sigma = 0.00023
theta = 0.000245

# Test different mu values to see how relationship changes with alpha
mu_values = np.array([0.00001, 0.00005, 0.0001, 0.001, 0.01, 0.05, 0.1])
x_values = np.linspace(theta - 0.002, theta + 0.002, 100)

# Collect all data
all_analytical = []
all_numerical = []
alpha_by_point = []

for mu in mu_values:
    alpha = (r / mu) - 1
    for x in x_values:
        try:
            numerical = stochastic.OrnsteinUhlenbeck.F(
                x, mu, sigma, theta, r, use_analytical=False
            )
            analytical = stochastic.OrnsteinUhlenbeck.F(
                x, mu, sigma, theta, r, use_analytical=True
            )

            # Convert to float if needed
            if isinstance(numerical, np.ndarray):
                numerical = float(numerical.item())
            if isinstance(analytical, np.ndarray):
                analytical = float(analytical.item())

            all_numerical.append(numerical)
            all_analytical.append(analytical)
            alpha_by_point.append(alpha)
        except Exception:
            pass

all_numerical = np.array(all_numerical)
all_analytical = np.array(all_analytical)
alpha_by_point = np.array(alpha_by_point)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "How Numerical vs Analytical Relationship Changes Across Inputs", fontsize=14
)

# Plot 1: Main scatter plot (like your example)
ax1 = axes[0, 0]
ax1.plot(
    all_numerical,
    all_analytical,
    "o",
    alpha=0.5,
    markersize=3,
    label="analytical vs numerical",
)
ax1.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax1.set_xlabel("Numerical F(x)")
ax1.set_ylabel("Analytical F(x)")
ax1.set_title("Analytical vs Numerical F(x)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Color-coded by alpha regime
ax2 = axes[0, 1]
scatter = ax2.scatter(
    all_numerical, all_analytical, c=alpha_by_point, cmap="viridis", alpha=0.6, s=20
)
ax2.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax2.set_xlabel("Numerical F(x)")
ax2.set_ylabel("Analytical F(x)")
ax2.set_title("Colored by α = r/μ - 1")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax2, label="α")

# Plot 3: Show relationship for different alpha regimes separately
ax3 = axes[1, 0]
colors = plt.cm.viridis(np.linspace(0, 1, len(mu_values)))
for i, mu in enumerate(mu_values):
    alpha = (r / mu) - 1
    mask = alpha_by_point == alpha
    if np.any(mask):
        ax3.plot(
            all_numerical[mask],
            all_analytical[mask],
            "o-",
            color=colors[i],
            alpha=0.6,
            markersize=2,
            label=f"α={alpha:.2f}",
        )

ax3.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax3.set_xlabel("Numerical F(x)")
ax3.set_ylabel("Analytical F(x)")
ax3.set_title("By Alpha Regime")
ax3.legend(fontsize=8, ncol=2, loc="upper left")
ax3.grid(True, alpha=0.3)

# Plot 4: Error analysis
ax4 = axes[1, 1]
errors = np.abs(all_analytical - all_numerical) / np.abs(all_numerical) * 100
ax4.scatter(all_numerical, errors, c=alpha_by_point, cmap="plasma", alpha=0.6, s=20)
ax4.set_xlabel("Numerical F(x)")
ax4.set_ylabel("Relative Error (%)")
ax4.set_title("Error vs F(x) Value (colored by α)")
ax4.set_yscale("log")
ax4.grid(True, alpha=0.3)
plt.colorbar(ax4.collections[0], ax=ax4, label="α")

plt.tight_layout()
plt.show()

# Print summary
print("\nSummary:")
print(f"  Total points: {len(all_numerical)}")
print(f"  Mean error: {np.mean(errors):.2f}%")
print(f"  Median error: {np.median(errors):.2f}%")
print(f"  Max error: {np.max(errors):.2f}%")
print(
    f"  Points with <1% error: {np.sum(errors < 1)} ({100 * np.sum(errors < 1) / len(errors):.1f}%)"
)
print(
    f"  Points with <5% error: {np.sum(errors < 5)} ({100 * np.sum(errors < 5) / len(errors):.1f}%)"
)

In [ ]:
plt.plot(numerical, label="numerical")
plt.plot(analytical, label="analytical")
plt.legend()
plt.show()

In [ ]:
pairs_trading_fitted_frame_computed["spread"] = (
    pairs_trading_fitted_frame_computed["close_1"]
    - pairs_trading_fitted_frame_computed["alpha"]
    - pairs_trading_fitted_frame_computed["beta"]
    * pairs_trading_fitted_frame_computed["close_2"]
)
pairs_trading_fitted_frame_computed

In [ ]:
# Compute entry and exit levels in parallel on resampled data
@delayed
def compute_entry_exit_levels_chunk(resampled_chunk: pd.DataFrame) -> pd.DataFrame:
    """
    Compute entry and exit levels for a chunk of resampled data.

    Args:
        resampled_chunk: DataFrame with columns: timestamp, provider_asset_group_id,
                        ou_mu, ou_theta, ou_sigma, and other OU parameters

    Returns:
        DataFrame with added columns: ou_L, entry_level, exit_level
    """
    result = resampled_chunk.copy()

    # Compute ou_L (loss level)
    result["ou_L"] = result["ou_theta"] - result["ou_sigma"] * STOP_LOSS_FACTOR

    # Initialize entry and exit levels
    result["entry_level"] = np.nan
    result["exit_level"] = np.nan

    # Compute entry and exit levels for rows with valid OU parameters
    valid_mask = (
        result["ou_mu"].notna()
        & result["ou_sigma"].notna()
        & result["ou_theta"].notna()
        & (result["ou_sigma"] > 0)
    )

    if valid_mask.any():
        valid_rows = result[valid_mask]

        for idx, row in valid_rows.iterrows():
            try:
                ou_mu = row["ou_mu"]
                ou_sigma = row["ou_sigma"]
                ou_theta = row["ou_theta"]
                ou_L = row["ou_L"]

                # Compute exit level first (needed for entry level)
                exit_level = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
                    mu=ou_mu,
                    sigma=ou_sigma,
                    theta=ou_theta,
                    r=DISCOUNT_RATE,
                    c=TRANSACTION_COST,
                    L=ou_L,
                )

                # Compute entry level (uses exit level internally)
                entry_level = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
                    mu=ou_mu,
                    sigma=ou_sigma,
                    theta=ou_theta,
                    r=DISCOUNT_RATE,
                    c=TRANSACTION_COST,
                    L=ou_L,
                    b_star=exit_level,  # Pass pre-computed exit level for efficiency
                )

                result.at[idx, "entry_level"] = entry_level
                result.at[idx, "exit_level"] = exit_level
            except Exception:
                # If computation fails, leave as NaN
                pass

    return result


# Create resampled data (similar to cell 16)
pairs_trading_resampled = (
    pairs_trading_fitted_frame_computed.drop(columns=["close_1", "close_2"])
    .set_index("timestamp")
    .groupby("provider_asset_group_id")
    .resample("1D")
    .first()
    .sort_values("timestamp")
    .drop(columns=["provider_asset_group_id"])
    .reset_index()
)

# Split into chunks for parallel processing
provider_asset_group_ids = pairs_trading_resampled["provider_asset_group_id"].unique()
n_chunks = min(N_WORKERS, len(provider_asset_group_ids))
group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

# Create delayed tasks
delayed_chunks = []
for chunk in group_chunks:
    chunk_data = pairs_trading_resampled[
        pairs_trading_resampled["provider_asset_group_id"].isin(chunk)
    ].copy()
    delayed_chunks.append(compute_entry_exit_levels_chunk(chunk_data))

# Compute in parallel
print("Computing entry and exit levels in parallel...")
pairs_trading_resampled_with_levels = pd.concat(
    [chunk.compute() for chunk in delayed_chunks], ignore_index=True
)
print(f"Computed levels for {len(pairs_trading_resampled_with_levels)} rows")
pairs_trading_resampled_with_levels

In [ ]:
# Join the original data with the resampled data that includes pre-computed entry/exit levels
pairs_trading_resampled_frame = pd.merge_asof(
    pairs_trading_fitted_frame_computed[
        ["timestamp", "provider_asset_group_id", "close_1", "close_2"]
    ]
    .drop_duplicates()
    .sort_values("timestamp"),
    pairs_trading_fitted_frame_computed.drop(
        columns=["close_1", "close_2"]
    ).sort_values("timestamp"),
    on="timestamp",
    by="provider_asset_group_id",
)
pairs_trading_resampled_frame

In [ ]:
pairs_trading_resampled_frame["spread"] = (
    pairs_trading_resampled_frame["close_1"]
    - pairs_trading_resampled_frame["alpha"]
    - pairs_trading_resampled_frame["beta"] * pairs_trading_resampled_frame["close_2"]
)
pairs_trading_resampled_frame

In [ ]:
pairs_trading_resampled_frame.loc[
    (pairs_trading_resampled_frame["p_value"] < 0.01)
    & (pairs_trading_resampled_frame["ou_sigma"] > 0.01)
]

In [ ]:
import plotly.graph_objects as go

example = 4070
example_df = pairs_trading_resampled_frame[
    pairs_trading_resampled_frame["provider_asset_group_id"] == example
].copy()

# Ensure timestamp is datetime for finer x-axis control
example_df["timestamp"] = pd.to_datetime(example_df["timestamp"])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=example_df["timestamp"], y=example_df["spread"], name="spread", mode="lines"
    )
)
fig.add_trace(
    go.Scatter(
        x=example_df["timestamp"],
        y=example_df["ou_theta"],
        name="ou_theta",
        mode="lines",
    )
)

# Add entry, exit, and loss levels if they exist
if "entry_level" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["entry_level"],
            name="entry_level",
            mode="lines",
            line=dict(color="green", width=1.5, dash="dash"),
        )
    )

if "exit_level" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["exit_level"],
            name="exit_level",
            mode="lines",
            line=dict(color="blue", width=1.5, dash="dash"),
        )
    )

if "ou_L" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["loss_level"],
            name="loss_level (ou_L)",
            mode="lines",
            line=dict(color="red", width=1.5, dash="dash"),
        )
    )

import numpy as np

n = len(example_df["timestamp"])
if n >= 8:
    tick_indices = np.linspace(0, n - 1, 8, dtype=int)
    tickvals = example_df["timestamp"].iloc[tick_indices]
else:
    tickvals = example_df["timestamp"]

# Format tick labels as nice datetimes
ticktext = [t.strftime("%Y-%m-%d %H:%M") for t in tickvals]

fig.update_layout(
    title=f"Spread, OU Theta, and Trading Levels for provider_asset_group_id {example}",
    xaxis_title="Timestamp",
    yaxis_title="Value",
    xaxis=dict(
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=45,  # Nicely angled for readability
    ),
)
fig.show()

In [ ]:
example_df["L"] = (
    example_df["residual_mean"] - example_df["residual_std"] * STOP_LOSS_FACTOR
)
example_dd

In [ ]:
cointegrated_provider_asset_group_ids = pairs_trading_fitted_frame_computed.loc[
    pairs_trading_fitted_frame_computed["p_value"] < 0.001
].index.tolist()
print(
    f"Cointegrated provider asset group ids (count: {len(cointegrated_provider_asset_group_ids)}): {cointegrated_provider_asset_group_ids}"
)

In [ ]:
def get_cointegrated_stats(df: pd.DataFrame) -> pd.Series:
    """
    Get the cointegrated stats for a given dataframe.
    """

    # Compute the linear regression.
    X = df["close_1"].to_numpy()
    y = df["close_2"].to_numpy()
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()

    # Get the residuals.
    linear_fit_alpha = results.params[0]
    linear_fit_beta = results.params[1]
    linear_fit_mse = results.mse_total
    linear_fit_r_squared = results.rsquared
    linear_fit_r_squared_adj = results.rsquared_adj
    residuals = results.resid

    # Get the cointegration stats.
    ou_params = OrnsteinUhlenbeck().fit(residuals)

    return pd.Series(
        [
            linear_fit_alpha,
            linear_fit_beta,
            linear_fit_mse,
            linear_fit_r_squared,
            linear_fit_r_squared_adj,
            ou_params.mu,
            ou_params.theta,
            ou_params.sigma,
        ],
        index=[
            "linear_fit_alpha",
            "linear_fit_beta",
            "linear_fit_mse",
            "linear_fit_r_squared",
            "linear_fit_r_squared_adj",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
        ],
        dtype=float,
    )

In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
cointegrated_pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    cointegrated_provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
cointegrated_pairs_trading_stats = cointegrated_pairs_trading_frame.groupby(
    "provider_asset_group_id"
)[["close_1", "close_2"]].apply(
    lambda df: get_cointegrated_stats(df),
    meta={
        "linear_fit_alpha": pd.Series([], dtype=float),
        "linear_fit_beta": pd.Series([], dtype=float),
        "linear_fit_mse": pd.Series([], dtype=float),
        "linear_fit_r_squared": pd.Series([], dtype=float),
        "linear_fit_r_squared_adj": pd.Series([], dtype=float),
        "ou_mu": pd.Series([], dtype=float),
        "ou_theta": pd.Series([], dtype=float),
        "ou_sigma": pd.Series([], dtype=float),
    },
)

In [ ]:
cointegrated_pairs_trading_stats_computed = cointegrated_pairs_trading_stats.compute()
cointegrated_pairs_trading_stats_computed

In [ ]:
# cointegration_p_values_computed.to_csv("cointegration_p_values.csv")
# cointegrated_pairs_trading_stats_computed.to_csv("cointegrated_pairs_trading_stats.csv")
# cointegration_p_values_computed.to_parquet("cointegration_p_values.parquet")
# cointegrated_pairs_trading_stats_computed.to_parquet(
#     "cointegrated_pairs_trading_stats.parquet"
# )


In [ ]:
toset = cointegration_p_values_computed.merge(
    cointegrated_pairs_trading_stats_computed, left_index=True, right_index=True
).reset_index()
toset = toset.rename(columns={"p_value": "cointegration_p_value"})
toset["lookback_window_seconds"] = 30 * 24 * 60 * 60
toset["timestamp"] = end_naive
toset = toset[
    [
        "timestamp",
        "provider_asset_group_id",
        "lookback_window_seconds",
        "cointegration_p_value",
        "linear_fit_alpha",
        "linear_fit_beta",
        "linear_fit_mse",
        "linear_fit_r_squared",
        "linear_fit_r_squared_adj",
        "ou_mu",
        "ou_theta",
        "ou_sigma",
    ]
]
toset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

data = (
    cointegrated_pairs_trading_frame.reset_index()
    .compute()
    .merge(
        toset.drop(columns=["lookback_window_seconds", "timestamp"]),
        on="provider_asset_group_id",
    )
)
data = data.loc[
    data["provider_asset_group_id"] == cointegrated_provider_asset_group_ids[0]
]
timestamps = data["timestamp"].to_numpy()
close_1 = data["close_1"].to_numpy()
close_2 = data["close_2"].to_numpy()
residuals = close_2 - data["linear_fit_alpha"] - data["linear_fit_beta"] * close_1

# Get the provider asset group members with asset symbols

from_asset = aliased(models.Asset)
to_asset = aliased(models.Asset)

pair_df = pd.read_sql(
    select(
        models.ProviderAssetGroupMember.order,
        from_asset.symbol.label("from_asset_symbol"),
        to_asset.symbol.label("to_asset_symbol"),
    )
    .join(from_asset, models.ProviderAssetGroupMember.from_asset_id == from_asset.id)
    .join(to_asset, models.ProviderAssetGroupMember.to_asset_id == to_asset.id)
    .where(
        models.ProviderAssetGroupMember.provider_asset_group_id
        == cointegrated_provider_asset_group_ids[0]
    ),
    engine,
)

# Get the symbols for the title
from_asset_1 = pair_df[pair_df["order"] == 1]["from_asset_symbol"].iloc[0]
from_asset_2 = pair_df[pair_df["order"] == 2]["from_asset_symbol"].iloc[0]
to_asset_1 = pair_df[pair_df["order"] == 1]["to_asset_symbol"].iloc[0]
to_asset_2 = pair_df[pair_df["order"] == 2]["to_asset_symbol"].iloc[0]

# Create figure and plot with nice date formatting
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(timestamps, residuals)
ax.set_title(
    f"Residuals for {to_asset_1}-{from_asset_1}/{to_asset_2}-{from_asset_2} (Pair ID: {cointegrated_provider_asset_group_ids[0]})"
)
ax.set_xlabel("Date")
ax.set_ylabel("Residuals")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
# set_data(
#     engine,
#     models.ProviderAssetGroupAttribute.__tablename__,
#     toset,
#     operation_type="upsert",
# )

In [ ]:
# cluster.close(force_shutdown=True)